In [1]:
### Cell 1: Imports ###
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from pathlib import Path
from torch.cuda.amp import autocast, GradScaler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
### Cell 2: Load and Preprocess Data ###
# UPDATED FOR 3D: Added 'z' to input and 'Uz' to output
df = pd.read_csv('/kaggle/input/trapezium3d-fluid-flow-simulation/concatenated_data_all_steps_3D.csv')
df_subset = df.groupby('time', group_keys=False).head(100)
print(f"Original shape: {df.shape}")
print(f"Subset shape: {df_subset.shape}")

# Optional — check how many unique time steps and how many samples per step
print(f"Unique time steps: {df_subset['time'].nunique()}")
print(df_subset['time'].value_counts().head())

X = df_subset[['time', 'x', 'y', 'z']].values
Y = df_subset[['Ux', 'Uy', 'Uz', 'p']].values

X_all = df[['time', 'x', 'y', 'z']].values
Y_all = df[['Ux', 'Uy', 'Uz', 'p']].values

y_scaler = StandardScaler()
Y_scaled = y_scaler.fit_transform(Y)
Y_scaled_all = y_scaler.transform(Y_all)

x_train_tensor = torch.tensor(X, dtype=torch.float32)
y_train_tensor = torch.tensor(Y_scaled, dtype=torch.float32)
y_all_tensor = torch.tensor(Y_scaled_all,dtype=torch.float32)

print(f"Input shape: {x_train_tensor.shape}")
print(f"Output shape: {y_train_tensor.shape}")

Original shape: (202000, 9)
Subset shape: (10100, 9)
Unique time steps: 101
time
10.0    100
0.0     100
0.1     100
0.2     100
0.3     100
Name: count, dtype: int64
Input shape: torch.Size([10100, 4])
Output shape: torch.Size([10100, 4])


In [3]:
### Cell 5: Define PDE Loss (Unsteady Laminar 3D Navier-Stokes) ###
nu = 5e-6 

# UPDATED FOR 3D: Added 'z' coordinate and 'w' (Uz) velocity
def pde_loss(t, x, y, z, pred):
    # u, v, w, p
    u, v, w, p = pred[:, 0:1], pred[:, 1:2], pred[:, 2:3], pred[:, 3:4]

    # First derivatives
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_y = torch.autograd.grad(u, y, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_z = torch.autograd.grad(u, z, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    
    v_t = torch.autograd.grad(v, t, grad_outputs=torch.ones_like(v), create_graph=True)[0]
    v_x = torch.autograd.grad(v, x, grad_outputs=torch.ones_like(v), create_graph=True)[0]
    v_y = torch.autograd.grad(v, y, grad_outputs=torch.ones_like(v), create_graph=True)[0]
    v_z = torch.autograd.grad(v, z, grad_outputs=torch.ones_like(v), create_graph=True)[0]

    w_t = torch.autograd.grad(w, t, grad_outputs=torch.ones_like(w), create_graph=True)[0]
    w_x = torch.autograd.grad(w, x, grad_outputs=torch.ones_like(w), create_graph=True)[0]
    w_y = torch.autograd.grad(w, y, grad_outputs=torch.ones_like(w), create_graph=True)[0]
    w_z = torch.autograd.grad(w, z, grad_outputs=torch.ones_like(w), create_graph=True)[0]
    p_x = torch.autograd.grad(p, x, grad_outputs=torch.ones_like(p), create_graph=True)[0]
    p_y = torch.autograd.grad(p, y, grad_outputs=torch.ones_like(p), create_graph=True)[0]
    p_z = torch.autograd.grad(p, z, grad_outputs=torch.ones_like(p), create_graph=True)[0]

    # Second derivatives (Laplacian)
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_zz = torch.autograd.grad(u_z, z, grad_outputs=torch.ones_like(u_z), create_graph=True)[0]
    
    v_xx = torch.autograd.grad(v_x, x, grad_outputs=torch.ones_like(v_x), create_graph=True)[0]
    v_yy = torch.autograd.grad(v_y, y, grad_outputs=torch.ones_like(v_y), create_graph=True)[0]
    v_zz = torch.autograd.grad(v_z, z, grad_outputs=torch.ones_like(v_z), create_graph=True)[0]

    w_xx = torch.autograd.grad(w_x, x, grad_outputs=torch.ones_like(w_x), create_graph=True)[0]
    w_yy = torch.autograd.grad(w_y, y, grad_outputs=torch.ones_like(w_y), create_graph=True)[0]
    w_zz = torch.autograd.grad(w_z, z, grad_outputs=torch.ones_like(w_z), create_graph=True)[0]

    # Continuity equation
    cont = u_x + v_y + w_z
    # Momentum equations
    # x-momentum
    f = u_t + (u * u_x + v * u_y + w * u_z) + p_x - nu * (u_xx + u_yy + u_zz)
    # y-momentum
    g = v_t + (u * v_x + v * v_y + w * v_z) + p_y - nu * (v_xx + v_yy + v_zz)
    # z-momentum
    h = w_t + (u * w_x + v * w_y + w * w_z) + p_z - nu * (w_xx + w_yy + w_zz)

    # Calculate losses
    f_loss = (f ** 2).mean()
    g_loss = (g ** 2).mean()
    h_loss = (h ** 2).mean()
    cont_loss = (cont ** 2).mean()
    
    return f_loss + g_loss + h_loss + cont_loss

In [4]:
## Cell 6: Define the Model (PirateNet) ###
# This class definition is generic and does not need changes
# The input_dim and output_dim are set during instantiation
class PirateNetBlock(nn.Module):
    def __init__(self, hidden_dim):
        super(PirateNetBlock, self).__init__()
        self.dense1 = nn.Linear(hidden_dim, hidden_dim)
        self.dense2 = nn.Linear(hidden_dim, hidden_dim)
        self.dense3 = nn.Linear(hidden_dim, hidden_dim)
        self.alpha = nn.Parameter(torch.zeros(1))

    def forward(self, x, u, v):
        f = F.tanh(self.dense1(x))
        z1 = f * u + (1 - f) * v
        g = F.tanh(self.dense2(z1))
        z2 = g * u + (1 - g) * v
        h = F.tanh(self.dense3(z2))
        return self.alpha * h + (1 - self.alpha) * x

class PirateNet(nn.Module):
    def __init__(
        self,
        input_dim,
        output_dim,
        num_blocks,
        hidden_dim=256,
        s=1.0,
        activation=F.tanh,
    ):
        super(PirateNet, self).__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.num_blocks = num_blocks
        self.hidden_dim = hidden_dim
        self.s = s
        self.activation = activation
        self.B = nn.Parameter(torch.randn(input_dim, hidden_dim // 2) * s)
        self.embedding = lambda x: torch.cat(
            [torch.cos(torch.matmul(x, self.B)), torch.sin(torch.matmul(x, self.B))],
            dim=-1,
        )
        self.blocks = nn.ModuleList(
            [PirateNetBlock(hidden_dim) for _ in range(num_blocks)]
        )
        self.U = nn.Linear(hidden_dim, hidden_dim)
        self.V = nn.Linear(hidden_dim, hidden_dim)
        self.final_layer = nn.Linear(hidden_dim, output_dim, bias=False)
        print(f"Final layer weights shape: {self.final_layer.weight.data.shape}")
        self.initialize_weights()

    def initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                torch.nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    module.bias.data.zero_()

    def forward(self, x):
        x = self.embedding(x)
        u = self.activation(self.U(x))
        v = self.activation(self.V(x))
        for block in self.blocks:
            x = block(x, u, v)
        return self.final_layer(x)

In [ ]:
scaler = GradScaler()

def train_model(model, optimizer, iterations, x_train, y_train, val_x, val_y):
    logs = {
        "iteration": [],
        "loss_from_pde": [],
        "loss_from_data": [],
        "total_loss": [],
        "val_loss": [],
    }
    best_val_loss = float('inf')
    patience = 50
    patience_counter = 0

    def closure():
        nonlocal best_val_loss, patience_counter
        optimizer.zero_grad()

        # Split input into individual tensors
        t, x, y, z = x_train[:, 0:1], x_train[:, 1:2], x_train[:, 2:3], x_train[:, 3:4]
        t.requires_grad_(True)
        x.requires_grad_(True)
        y.requires_grad_(True)
        z.requires_grad_(True)
        input_tensor = torch.cat([t, x, y, z], dim=1)

        # Forward pass under autocast (mixed precision)
        with autocast():
            pred = model(input_tensor)
            loss_from_data = F.mse_loss(pred, y_train)

        # PDE loss in full precision (for gradient stability)
        loss_from_pde = pde_loss(t, x, y, z, pred.float())

        # Combine losses
        loss = loss_from_data + loss_from_pde

        # Backward with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Validation step
        model.eval()
        with torch.no_grad():
            val_pred = model(val_x)
            val_loss = F.mse_loss(val_pred, val_y)
        model.train()

        # Logging
        closure.iteration += 1
        logs["iteration"].append(closure.iteration)
        logs["loss_from_pde"].append(loss_from_pde.item())
        logs["loss_from_data"].append(loss_from_data.item())
        logs["total_loss"].append(loss.item())
        logs["val_loss"].append(val_loss.item())

        # Print progress
        if closure.iteration % 100 == 0:
            print(
                f"Iter {closure.iteration:04d}: PDE={loss_from_pde.item():.3f}, "
                f"Data={loss_from_data.item():.3f}, Val={val_loss.item():.3f}"
            )

        # Early stopping
        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model_adam.pth')
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping at iteration {closure.iteration}")
            raise KeyboardInterrupt

        return loss

    closure.iteration = 0

    try:
        for iteration in range(iterations):
            _ = closure()
    except KeyboardInterrupt:
        print("Training stopped early or manually.")

    model.load_state_dict(torch.load('best_model_adam.pth'))
    return logs


### Cell 8: Setup Training ###
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(42)
indices = torch.randperm(x_train_tensor.size(0))
train_size = int(0.8 * x_train_tensor.size(0))

x_train = x_train_tensor[indices[:train_size]].to(device)
y_train = y_train_tensor[indices[:train_size]].to(device)
x_val = x_train_tensor[indices[train_size:]].to(device)
y_val = y_train_tensor[indices[train_size:]].to(device)

model = PirateNet(
    input_dim=4,   # (t, x, y, z)
    output_dim=4,  # (Ux, Uy, Uz, p)
    num_blocks=8,
    hidden_dim=256,
).to(device)


### Cell 9: Train with Adam ###
lr = 0.001
iterations = 3000
optimizer_adam = optim.Adam(model.parameters(), lr=lr)

logs_adam = train_model(model, optimizer_adam, iterations, x_train, y_train, x_val, y_val)

/tmp/ipykernel_37/168646533.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Using device: cuda
Final layer weights shape: torch.Size([4, 256])


/tmp/ipykernel_37/168646533.py:28: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Iter 0100: PDE=1.674e-02, Data=5.983e-01, Val=5.862e-01
Iter 0200: PDE=9.209e-02, Data=3.610e-01, Val=3.514e-01
Iter 0300: PDE=1.183e-02, Data=3.036e-01, Val=3.006e-01
Iter 0400: PDE=1.172e-01, Data=2.727e-01, Val=2.727e-01
